# Notebook 01 - Data layer (R1)

Loads the US 1980 input-output tables from `BFdata.csv`, replicates the MATLAB
cleaning (remove government sectors 60, 80:88 and zero-sales sectors 8, 62),
and builds the primitives Omega, alpha, beta, lambda, L.

Exports a small summary CSV (`data_summary.csv`) for human inspection.

In [ ]:
# --- Project setup (robust path resolution) ---
# @__DIR__ resolves to this notebook's directory; we anchor on the package root.
using LinearAlgebra, Statistics, Printf, DelimitedFiles

const NOTEBOOK_DIR = @__DIR__
const REP_DIR = joinpath(NOTEBOOK_DIR, "..")          # bf_replication/
cd(REP_DIR)
include(joinpath(REP_DIR, "src", "BFReplication.jl"))
using .BFReplication
using .BFReplication.DataLoader
using .BFReplication.BFModel
using .BFReplication.InflationAnalysis

const DATA_DIR = joinpath(REP_DIR, "..", "Replication Files", "GDP Simulatin -- 88 Sector")
const RESULTS_DIR = joinpath(REP_DIR, "data", "results")
mkpath(RESULTS_DIR)

println("Project dir : ", REP_DIR)
println("Data dir    : ", DATA_DIR)
println("Results dir : ", RESULTS_DIR)

## Load + describe

In [ ]:
data = load_bf_data(joinpath(DATA_DIR, "BFdata.csv"); year=1980)
describe_data(data)

## Sanity assertions

In [ ]:
@assert sum(data.β) ≈ 1.0                    "β must sum to 1"
@assert maximum(abs.(data.λ .* data.α .- data.L)) < 1e-10 "λα must equal L"
@assert data.N == 76                          "expected 76 sectors after removals"
println("Sanity checks passed: N = $(data.N), sum(β) = $(round(sum(data.β), digits=12))")

## Export a summary CSV (sector-level shares)

In [ ]:
open(joinpath(RESULTS_DIR, "data_summary.csv"), "w") do io
    println(io, "sector,alpha,beta,lambda,L")
    for i in 1:data.N
        println(io, i, ",", data.α[i], ",", data.β[i], ",", data.λ[i], ",", data.L[i])
    end
end
println("wrote data_summary.csv")